# Public Wafer Pattern Evidence

Reproduced selection, grouped confirmation, class-tail, ONNX runtime, and lifecycle evidence for the independently generated synthetic wafer-pattern benchmark. This notebook does not access WM-811K, private data, or STDF input.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "evidence").is_dir() and (ROOT.parent / "evidence").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "evidence").is_dir(), "Run from the project root or notebooks directory"
sys.path.insert(0, str(ROOT))

def load_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))

evaluation = load_json("evidence/public_synthetic_evaluation.json")
metadata = load_json("models/public_synthetic_resnet18_v1.json")
claims = load_json("evidence/claims.json")
print(f"Project root: {ROOT.name}")
print(f"Evidence scope: {evaluation['data_scope']}")
print(f"Model: {metadata['model_version']}")

Project root: 04_Transfer_Learning_ResNet_STDF_Wafer_Map_Yield_Predictor
Evidence scope: independently generated grouped synthetic wafer maps
Model: public_synthetic_resnet18_v1


In [2]:
metrics = evaluation["confirmation_metrics"]
pd.DataFrame({
    "Metric": ["Samples", "Accuracy", "Macro F1", "Balanced accuracy", "MCC", "Macro PR-AUC", "Top-label ECE"],
    "Value": [metrics["samples"], metrics["accuracy"], metrics["macro_f1"], metrics["balanced_accuracy"], metrics["mcc"], metrics["macro_pr_auc"], metrics["top_label_ece"]],
})

,Metric,Value
0,Samples,800.000000
1,Accuracy,0.936250
2,Macro F1,0.936073
3,Balanced accuracy,0.936250
4,MCC,0.927279
5,Macro PR-AUC,0.980099
6,Top-label ECE,0.027026


In [3]:
dataset = evaluation["dataset"]
print(f"Train / validation / confirmation: {dataset['train_samples']:,} / {dataset['validation_samples']:,} / {dataset['confirmation_samples']:,}")
print(f"Selected C: {evaluation['selected_c']}")
print(f"Validation-only temperature: {evaluation['temperature']:.5f}")
print(f"Group overlap counts: {evaluation['group_overlap']}")
print(f"Confirmation opened after selection freeze: {evaluation['confirmation_opened_after_selection_freeze']}")

Train / validation / confirmation: 1,920 / 480 / 800
Selected C: 0.1
Validation-only temperature: 0.71947
Group overlap counts: {'train_validation': 0, 'train_confirmation': 0, 'validation_confirmation': 0}
Confirmation opened after selection freeze: True


In [4]:
class_rows = [
    {"Class": class_name, **class_metrics}
    for class_name, class_metrics in metrics["per_class"].items()
]
class_table = pd.DataFrame(class_rows).sort_values("recall")
print(f"Weakest class: {class_table.iloc[0]['Class']} at {class_table.iloc[0]['recall']:.1%} recall")
class_table

Weakest class: QuadrantFailure at 83.0% recall


,Class,precision,recall,f1,support
4,QuadrantFailure,0.882979,0.83,0.855670,100
0,Normal,0.967391,0.89,0.927083,100
5,Scratch,0.850467,0.91,0.879227,100
1,EdgeEffect,0.893204,0.92,0.906404,100
2,CenterCluster,0.950000,0.95,0.950000,100
6,RandomFailure,0.980198,0.99,0.985075,100
3,RingPattern,0.980392,1.00,0.990099,100
7,MixedMode,0.990099,1.00,0.995025,100


In [5]:
from src.api.inference import file_sha256, verify_public_artifact

verified = verify_public_artifact()
print(f"ONNX SHA-256: {verified['model_sha256']}")
print(f"Metadata SHA-256: {verified['metadata_sha256']}")
print(f"Manifest SHA-256: {metadata['dataset_manifest_sha256']}")
print(f"All promotion gates passed: {evaluation['passes_all_gates']}")

ONNX SHA-256: fca5a3dea9ec9b019476712bed3ead758e9ec5a290508836a21f669d7a13dcb6
Metadata SHA-256: 6041c4648963211a4601bd84099af7530948d91faf4c8ae226f7306432a763fb
Manifest SHA-256: b8b93f97cd2a7897873c7e091a9a5e2359f6448ca1227bbcc9671d64df7e03fc
All promotion gates passed: True


In [6]:
from src.api.inference import ModelInference
from src.data.synthetic_benchmark import build_sample_specs, generate_wafer_image

confirmation_spec = next(
    spec for spec in build_sample_specs(variants_per_family=1)
    if spec.split == "confirmation" and spec.class_name == "RingPattern"
)
prediction = ModelInference().predict(generate_wafer_image(confirmation_spec))
print(json.dumps({
    "generated_class": confirmation_spec.class_name,
    "predicted_class": prediction["defect_class"],
    "confidence": prediction["confidence"],
    "model_version": prediction["model_version"],
}, indent=2))

{
  "generated_class": "RingPattern",
  "predicted_class": "RingPattern",
  "confidence": 0.9996011853218079,
  "model_version": "public_synthetic_resnet18_v1"
}


In [7]:
unsupported = [
    claim["claim"]
    for claim in claims["claims"]
    if claim["evidence_class"] == "unsupported"
]
print("Implemented: grouped synthetic generator, validation-only selection, disjoint confirmation, hash-verified ONNX image API.")
print(f"Unsupported public claims: {unsupported}")
print("Boundary: no WM-811K proof, no STDF parsing claim, no yield prediction, no production silicon outcome.")

Implemented: grouped synthetic generator, validation-only selection, disjoint confirmation, hash-verified ONNX image API.
Unsupported public claims: ['WM-811K test accuracy', 'STDF-to-classification pipeline', 'Wafer yield prediction']
Boundary: no WM-811K proof, no STDF parsing claim, no yield prediction, no production silicon outcome.
